# 企业名称模糊匹配 — 强化实战

**目标**：掌握「规则预处理 → 候选集缩小 → LLM精判 → 人工复核」混合匹配方法。
**实战场景**：将AI舞弊标注数据中的上市公司名称对齐到CSMAR/Wind财务数据库。


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import logging; logging.basicConfig(level=logging.ERROR)

import re, json, time, os
import requests
import pandas as pd
import numpy as np
from collections import Counter

print('环境就绪')


## 任务一：课堂练习 — 供应商名称匹配

10 条业务系统供应商 vs 13 条标准供应商主数据。


In [ ]:
# 课堂练习数据
to_match = pd.DataFrame([
    {'record_id': 'R001', 'raw_name': '上海晨光文具', 'scene': '办公用品采购'},
    {'record_id': 'R002', 'raw_name': '北京德邦物流有限公司', 'scene': '物流运输'},
    {'record_id': 'R003', 'raw_name': '华为技术', 'scene': '设备采购'},
    {'record_id': 'R004', 'raw_name': '深圳顺丰速运', 'scene': '物流运输'},
    {'record_id': 'R005', 'raw_name': '杭州阿里巴巴', 'scene': '技术服务'},
    {'record_id': 'R006', 'raw_name': '广州宝洁有限公司', 'scene': '办公用品采购'},
    {'record_id': 'R007', 'raw_name': '中石化', 'scene': '能源采购'},
    {'record_id': 'R008', 'raw_name': '北京京东世纪贸易', 'scene': '设备采购'},
    {'record_id': 'R009', 'raw_name': '腾讯科技深圳', 'scene': '技术服务'},
    {'record_id': 'R010', 'raw_name': '上海复星医药集团', 'scene': '医疗采购'},
])

standard = pd.DataFrame([
    {'std_id': 'S01', 'std_name': '上海晨光文具股份有限公司', 'type': '办公用品'},
    {'std_id': 'S02', 'std_name': '德邦物流股份有限公司北京分公司', 'type': '物流运输'},
    {'std_id': 'S03', 'std_name': '华为技术有限公司', 'type': '设备'},
    {'std_id': 'S04', 'std_name': '顺丰速运有限公司', 'type': '物流运输'},
    {'std_id': 'S05', 'std_name': '阿里巴巴（中国）有限公司', 'type': '技术服务'},
    {'std_id': 'S06', 'std_name': '广州宝洁有限公司', 'type': '办公用品'},
    {'std_id': 'S07', 'std_name': '中国石油化工集团有限公司', 'type': '能源'},
    {'std_id': 'S08', 'std_name': '北京京东世纪贸易有限公司', 'type': '设备'},
    {'std_id': 'S09', 'std_name': '深圳市腾讯计算机系统有限公司', 'type': '技术服务'},
    {'std_id': 'S10', 'std_name': '上海复星医药（集团）股份有限公司', 'type': '医疗'},
    {'std_id': 'S11', 'std_name': '用友网络科技股份有限公司', 'type': '软件'},
    {'std_id': 'S12', 'std_name': '浪潮电子信息产业股份有限公司', 'type': '设备'},
    {'std_id': 'S13', 'std_name': '东软集团股份有限公司', 'type': '软件'},
])

display(to_match); display(standard)


## 清洗函数：名称标准化


In [ ]:
def clean_name(name):
    """清洗公司名称，去除干扰信息，保留核心词。"""
    if pd.isna(name): return ''
    name = str(name).strip()
    # 去除括号内容
    name = re.sub(r'[（\(].*?[）\)]', '', name)
    # 去除通用后缀（保留核心区分词）
    suffixes = ['股份有限公司', '有限责任公司', '有限公司', '集团', '控股', '科技', '技术', '贸易', '实业']
    for s in sorted(suffixes, key=len, reverse=True):
        name = name.replace(s, '')
    # 去除常见地名前缀
    cities = ['深圳市', '深圳', '北京市', '北京', '上海市', '上海', '广州市', '广州', '杭州市', '杭州']
    for c in sorted(cities, key=len, reverse=True):
        if name.startswith(c):
            name = name[len(c):]
            break
    return name.strip()

# 测试
for n in ['上海晨光文具股份有限公司', '深圳市腾讯计算机系统有限公司', '华为技术有限公司']:
    print(f'{n} → {clean_name(n)}')


## 候选集缩小：字符重合度 + 行业匹配


In [ ]:
def get_candidates(query, candidates_df, top_k=5):
    """用字符重合度 + 行业加分缩小候选集"""
    q_clean = clean_name(query)
    q_chars = set(q_clean)
    
    scored = []
    for _, row in candidates_df.iterrows():
        c_clean = clean_name(row['std_name'])
        c_chars = set(c_clean)
        if not q_chars or not c_chars:
            scored.append((row['std_id'], row['std_name'], 0))
            continue
        # Jaccard 相似度
        overlap = len(q_chars & c_chars)
        union = len(q_chars | c_chars)
        score = overlap / union if union > 0 else 0
        # 行业加分
        if 'scene' in candidates_df.columns and 'type' in candidates_df.columns:
            if row.get('type', '') in query or True:
                pass  # 简化处理
        scored.append((row['std_id'], row['std_name'], round(score, 4)))
    scored.sort(key=lambda x: x[2], reverse=True)
    return pd.DataFrame(scored[:top_k], columns=['std_id', 'std_name', 'score'])

# 测试
print('查询: 上海晨光文具')
print(get_candidates('上海晨光文具', standard, top_k=3).to_string(index=False))


## LLM 精确匹配（MiniMax API）


In [ ]:
MINIMAX_URL = 'https://api.minimaxi.com/anthropic/v1/messages'
MINIMAX_KEY = 'sk-api-mpG58Vfbq_73kWdXkfCEAM1_PV4n6ehehBArCmHaVAUGeUNPTp_eWA2NcJwdt-sTkpPNmh4ulLNBCU1yYI-G61c0iUlVA6805UhKxgaTOO2fqOYIUgdLheI'

def llm_match(query, candidates_df):
    """用 LLM 在候选集中精确判断匹配"""
    cand_list = candidates_df[['std_id', 'std_name']].to_dict(orient='records')
    prompt = f'''请判断查询名称最可能对应候选列表中的哪一个。

查询名称：{query}
候选列表：{json.dumps(cand_list, ensure_ascii=False, indent=2)}

规则：1.可识别简称、地名省略、后缀差异 2.无法判断时 matched_id 为 null
输出纯JSON：{{"matched_id":"Sxx或null","confidence":0-1,"reason":"判断理由"}}'''
    
    r = requests.post(MINIMAX_URL, json={
        'model': 'MiniMax-M2.7', 'max_tokens': 500,
        'system': '你是企业名称匹配专家。只输出JSON。',
        'messages': [{'role': 'user', 'content': prompt}],
        'thinking': {'type': 'disabled'},
    }, headers={
        'x-api-key': MINIMAX_KEY, 'anthropic-version': '2023-06-01', 'content-type': 'application/json'
    }, timeout=30)
    
    if r.status_code != 200:
        return {'matched_id': None, 'confidence': 0, 'reason': f'API错误:{r.status_code}'}
    
    # 找text块
    for block in r.json().get('content', []):
        if block.get('type') == 'text':
            try:
                text = block['text'].strip()
                # 去除 markdown 包裹
                m = re.search(r'\{[^\}]+\}', text, re.DOTALL)
                if m: return json.loads(m.group())
            except: pass
    return {'matched_id': None, 'confidence': 0, 'reason': '解析失败'}

print('LLM匹配函数就绪')


## 执行全量匹配


In [ ]:
results = []
for _, row in to_match.iterrows():
    name = row['raw_name']
    candidates = get_candidates(name, standard, top_k=3)
    match = llm_match(name, candidates)
    
    # 查找匹配的标准名称
    matched_name = None
    if match.get('matched_id'):
        m = standard[standard['std_id'] == match['matched_id']]
        if len(m) > 0: matched_name = m.iloc[0]['std_name']
    
    results.append({
        'record_id': row['record_id'],
        'raw_name': name,
        'scene': row['scene'],
        'matched_name': matched_name,
        'confidence': match.get('confidence', 0),
        'reason': match.get('reason', ''),
        'candidates': ' | '.join(candidates['std_name'].tolist()),
    })
    print(f'{name} → {matched_name or "未匹配"} (置信度:{match.get("confidence",0)})')

df_result = pd.DataFrame(results)


## 人工复核标记


In [ ]:
def classify(row):
    if pd.isna(row['matched_name']) or row['confidence'] < 0.55:
        return '🔴 需人工处理'
    elif row['confidence'] < 0.85:
        return '🟡 建议复核'
    return '🟢 高置信通过'

df_result['status'] = df_result.apply(classify, axis=1)
print('=== 匹配结果 ===')
display(df_result[['raw_name', 'matched_name', 'confidence', 'status']])

print(f"\n高置信通过: {(df_result['status']=='🟢 高置信通过').sum()}/10")
print(f"建议复核: {(df_result['status']=='🟡 建议复核').sum()}/10")
print(f"需人工: {(df_result['status']=='🔴 需人工处理').sum()}/10")


## 任务二：实战 — 标注数据公司名匹配

从 AI 舞弊标注结果中提取去重后的上市公司名，为对接 CSMAR 做准备。


In [ ]:
# 加载标注数据
labeled = pd.read_excel('../../智能财务前沿/STK_labeled_何其轩_2023111180_G04.xlsx')
print(f'标注数据: {len(labeled)} 行')

# 提取去重公司
companies = labeled[['Symbol', 'ShortName', 'CoFullName']].drop_duplicates()
print(f'去重公司: {len(companies)} 家')

# 查看分布
print(f'Symbol 前缀分布:')
print(companies['Symbol'].str[:3].value_counts().head(10))
display(companies.head(10))


## 总结

**混合匹配方法论**：
1. 清洗：去除括号、后缀、地名前缀 → 保留核心词
2. 候选集缩小：Jaccard 字符重合度 + 行业/地域匹配 → Top-3/5
3. LLM 精判：在 3-5 个候选中做精确判断 → 置信度 + 理由
4. 人工复核：conf < 0.85 → 标记复核

**对阶段二的价值**：
- 公司名匹配是连接「处罚书数据」和「CSMAR 财务数据」的桥梁
- Symbol（股票代码）是主键，但退市/改名/重组后可能失效
- CoFullName → 数据库公司名的模糊匹配作为兜底方案
- 匹配率直接影响可用样本量，进而影响模型效果
